In [ ]:
from enum import Enum
from datetime import datetime
import os


# =========================
# CONFIGURACIÓN
# =========================

RUTA_BASE = "SGH_datos/"

if not os.path.exists(RUTA_BASE):
    os.makedirs(RUTA_BASE)

RUTA_PACIENTES = RUTA_BASE + "pacientes.txt"
RUTA_MEDICOS = RUTA_BASE + "medicos.txt"
RUTA_ESPECIALIDADES = RUTA_BASE + "especialidades.txt"
RUTA_CITAS = RUTA_BASE + "citas.txt"


# =========================
# ENUMS
# =========================

class RegimenEnum(Enum):
    CONTRIBUTIVO = "CONTRIBUTIVO"
    SUBSIDIADO = "SUBSIDIADO"


class EstadoCitaEnum(Enum):
    PROGRAMADA = "PROGRAMADA"
    ATENDIDA = "ATENDIDA"
    CANCELADA = "CANCELADA"


# =========================
# MODELO / DOMINIO
# =========================

class Paciente:
    def __init__(self, num_documento, nombre, fecha_nacimiento, tipo_sangre, eps, regimen, antecedentes):
        self.num_documento = num_documento
        self.nombre = nombre
        self.fecha_nacimiento = fecha_nacimiento
        self.tipo_sangre = tipo_sangre
        self.eps = eps
        self.regimen = regimen
        self.antecedentes = antecedentes

    def to_linea(self):
        return f"{self.num_documento}|{self.nombre}|{self.fecha_nacimiento}|{self.tipo_sangre}|{self.eps}|{self.regimen}|{self.antecedentes}"

    @staticmethod
    def from_linea(linea):
        datos = linea.strip().split("|")
        return Paciente(datos[0], datos[1], datos[2], datos[3], datos[4], datos[5], datos[6])


class Medico:
    def __init__(self, num_registro, nombre, especialidad, consultorio, horario):
        self.num_registro = num_registro
        self.nombre = nombre
        self.especialidad = especialidad
        self.consultorio = consultorio
        self.horario = horario

    def to_linea(self):
        return f"{self.num_registro}|{self.nombre}|{self.especialidad}|{self.consultorio}|{self.horario}"


class Especialidad:
    def __init__(self, codigo, nombre, descripcion):
        self.codigo = codigo
        self.nombre = nombre
        self.descripcion = descripcion

    def to_linea(self):
        return f"{self.codigo}|{self.nombre}|{self.descripcion}"


class Cita:
    def __init__(self, codigo, num_documento_paciente, num_registro_medico, fecha, hora, motivo, estado):
        self.codigo = codigo
        self.num_documento_paciente = num_documento_paciente
        self.num_registro_medico = num_registro_medico
        self.fecha = fecha
        self.hora = hora
        self.motivo = motivo
        self.estado = estado

    def to_linea(self):
        return f"{self.codigo}|{self.num_documento_paciente}|{self.num_registro_medico}|{self.fecha}|{self.hora}|{self.motivo}|{self.estado}"


# =========================
# REPOSITORIOS
# =========================

class ArchivoUtil:
    @staticmethod
    def leer_lineas(ruta):
        try:
            with open(ruta, "r", encoding="utf-8") as archivo:
                return [linea.strip() for linea in archivo.readlines()]
        except FileNotFoundError:
            return []

    @staticmethod
    def escribir_linea(ruta, linea):
        with open(ruta, "a", encoding="utf-8") as archivo:
            archivo.write(linea + "\n")

    @staticmethod
    def reescribir(ruta, lineas):
        with open(ruta, "w", encoding="utf-8") as archivo:
            for linea in lineas:
                archivo.write(linea + "\n")

    @staticmethod
    def generar_codigo(prefijo):
        fecha = datetime.now().strftime("%Y%m%d%H%M%S")
        return f"{prefijo}-{fecha}"


class PacienteRepository:
    def guardar(self, paciente):
        ArchivoUtil.escribir_linea(RUTA_PACIENTES, paciente.to_linea())

    def listar_todos(self):
        lineas = ArchivoUtil.leer_lineas(RUTA_PACIENTES)
        return [Paciente.from_linea(linea) for linea in lineas if linea]

    def buscar_por_documento(self, documento):
        pacientes = self.listar_todos()
        for paciente in pacientes:
            if paciente.num_documento == documento:
                return paciente
        return None


# =========================
# SERVICIOS
# =========================

class PacienteService:
    def __init__(self):
        self.repo = PacienteRepository()

    def registrar_paciente(self, paciente):
        existente = self.repo.buscar_por_documento(paciente.num_documento)

        if existente:
            print("Ya existe un paciente con ese documento.")
        else:
            self.repo.guardar(paciente)
            print("Paciente registrado correctamente.")

    def consultar_paciente(self, documento):
        paciente = self.repo.buscar_por_documento(documento)

        if paciente:
            print("\n--- DATOS DEL PACIENTE ---")
            print("Documento:", paciente.num_documento)
            print("Nombre:", paciente.nombre)
            print("Fecha de nacimiento:", paciente.fecha_nacimiento)
            print("Tipo de sangre:", paciente.tipo_sangre)
            print("EPS:", paciente.eps)
            print("Régimen:", paciente.regimen)
            print("Antecedentes:", paciente.antecedentes)
        else:
            print("Paciente no encontrado.")

    def listar_pacientes(self):
        pacientes = self.repo.listar_todos()

        if not pacientes:
            print("No hay pacientes registrados.")
        else:
            print("\n--- LISTADO DE PACIENTES ---")
            for p in pacientes:
                print(f"{p.num_documento} - {p.nombre} - {p.eps}")


# =========================
# INTERFAZ DE USUARIO
# =========================

def menu_pacientes():
    servicio = PacienteService()

    while True:
        print("\n===== MENÚ PACIENTES =====")
        print("1. Registrar paciente")
        print("2. Consultar paciente")
        print("3. Listar pacientes")
        print("0. Volver")

        opcion = input("Seleccione una opción: ")

        if opcion == "1":
            documento = input("Número de documento: ")
            nombre = input("Nombre completo: ")
            fecha_nacimiento = input("Fecha de nacimiento AAAA-MM-DD: ")
            tipo_sangre = input("Tipo de sangre: ")
            eps = input("EPS: ")
            regimen = input("Régimen CONTRIBUTIVO/SUBSIDIADO: ")
            antecedentes = input("Antecedentes médicos: ")

            paciente = Paciente(
                documento,
                nombre,
                fecha_nacimiento,
                tipo_sangre,
                eps,
                regimen,
                antecedentes
            )

            servicio.registrar_paciente(paciente)

        elif opcion == "2":
            documento = input("Ingrese documento del paciente: ")
            servicio.consultar_paciente(documento)

        elif opcion == "3":
            servicio.listar_pacientes()

        elif opcion == "0":
            break

        else:
            print("Opción inválida.")


def menu_principal():
    while True:
        print("\n====== SISTEMA DE GESTIÓN HOSPITALARIA ======")
        print("1. Gestión de pacientes")
        print("2. Gestión de médicos")
        print("3. Gestión de citas")
        print("4. Registro clínico")
        print("5. Análisis de datos")
        print("0. Salir")

        opcion = input("Seleccione una opción: ")

        if opcion == "1":
            menu_pacientes()
        elif opcion in ["2", "3", "4", "5"]:
            print("Módulo pendiente por implementar en el siguiente avance.")
        elif opcion == "0":
            print("Saliendo del sistema...")
            break
        else:
            print("Opción inválida.")


# =========================
# EJECUCIÓN PRINCIPAL
# =========================

menu_principal()

KeyboardInterrupt: Interrupted by user